# Catholic Public Domain Bible version updater

By Kenneth Burchfiel

Released under the MIT License

This notebook is designed to make it easier to update an existing set of test results with new verses. (These new verses might include certain corrections, e.g. trailing whitespace removals, or updates from the sacredbible.org website.)

**Prerequisites**:
1. Copy your existing CPDB_for_TTTB.csv file (e.g. the one that contains your test results) into the CPDB_Version_Updater folder that stores this project, then **rename it** Existing_CPDB_for_TTTB.csv.
2. Copy the new CPDB_for_TTTB.csv file (which has the updated verses that you'd like to incorporate into this project) into this folder, then **rename it**s New_CPDB_for_TTTB.csv.

Note: I recommend running this code step by step in order to check the output; because it relies on each verse ID corresponding to the same original verse, any change in order could cause all test result data to get removed from the new file.

**After running the script**:
1. Look over the CPDB_for_TTTB.csv file created by this code to make sure it still looks correct.
2. Consider making a copy of your existing CPDB_for_TTTB.csv file so that you can revert to it if you encounter any issues with the new one. (Or simply keep the Existing_CPDB_for_TTTB.csv file that you now have in this folder.)
3. If you're ready to replace the older version with the newer one, rename 'New_CPDB_for_TTTB.csv' as 'CPDB_for_TTTB.csv', then copy and paste it into your /Files/ folder (thus overwriting the older version.)

In [1]:
import time
start_time = time.time()
import pandas as pd
import numpy as np
from thefuzz import fuzz

## Importing the two files:

In [2]:
df_existing = pd.read_csv('Existing_CPDB_for_TTTB.csv')
# df_existing['Verse_ID'] = df_existing['Verse_ID'][::-1].to_list() 
# For debugging purposes only! This line can help you preview what the 
# output will look like if the verse IDs get out of sync with one another.
df_new = pd.read_csv('New_CPDB_for_TTTB.csv')
headers = df_new.columns.to_list() # These should be the same for both 
# files; if they differ, the script will raise an error shortly.

### Making sure both files have the same length and headers: 

(This will indicate, but not guarantee, that the two files' Verse_ID values--our merge key--will be the same.)

In [3]:
if len(df_existing) != len(df_new):
    raise ValueError("Files' row counts differ! You may need to instead \
make manual updates to the existing document in order to prevent \
different verses from getting paired together.")
if list(df_new.columns) != list(df_existing.columns):
    raise ValueError("Files have different headers! A manual update may \
be required instead.")

In [4]:
# We'll retain df_existing's Tests and Best_WPM columns for those verses
# that match their counterparts within df_new. (If verses have instead
# been changed, we'll clear out this data, as the new verse might differ
# substantially from the old one and thus render this data invalid.)
df_existing_for_merge = df_existing.copy()[
['Verse_ID', 'Verse', 'Tests', 'Best_WPM']].rename(
columns={'Verse':'Former_Verse'})

df_existing_for_merge

,Verse_ID,Former_Verse,Tests,Best_WPM
0,1,"In the beginning, God created heaven and earth.",29,200.543770
1,2,"But the earth was empty and unoccupied, and da...",7,161.088381
2,3,"And God said, ""Let there be light."" And light ...",7,157.810945
3,4,"And God saw the light, that it was good; and s...",5,162.812696
4,5,"And he called the light, 'Day,' and the darkne...",2,130.008479
...,...,...,...,...
35834,35835,"And the Spirit and the Bride say: ""Draw near.""...",0,0.000000
35835,35836,For I call as witnesses all listeners of the w...,0,0.000000
35836,35837,And if anyone will have taken away from the wo...,2,136.493900
35837,35838,"He who offers testimony to these things, says:...",2,125.198421


In [5]:
# Meanwhile, verse data will come from df_new.
df_new_for_merge = df_new.copy()[[
'Verse_ID', 'OT_NT', 'Book', 'Book_Num', 'Chapter_Num', 
'Verse_Num', 'Verse_Code', 'Verse', 'Characters']]
df_new_for_merge

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters
0,1,OT,Genesis,1,1,1,Genesis_1:1,"In the beginning, God created heaven and earth.",47
1,2,OT,Genesis,1,1,2,Genesis_1:2,"But the earth was empty and unoccupied, and da...",141
2,3,OT,Genesis,1,1,3,Genesis_1:3,"And God said, ""Let there be light."" And light ...",53
3,4,OT,Genesis,1,1,4,Genesis_1:4,"And God saw the light, that it was good; and s...",89
4,5,OT,Genesis,1,1,5,Genesis_1:5,"And he called the light, 'Day,' and the darkne...",104
...,...,...,...,...,...,...,...,...,...
35834,35835,NT,Revelation,73,22,17,Revelation_22:17,"And the Spirit and the Bride say: ""Draw near.""...",197
35835,35836,NT,Revelation,73,22,18,Revelation_22:18,For I call as witnesses all listeners of the w...,176
35836,35837,NT,Revelation,73,22,19,Revelation_22:19,And if anyone will have taken away from the wo...,217
35837,35838,NT,Revelation,73,22,20,Revelation_22:20,"He who offers testimony to these things, says:...",108


In [6]:
df_updated = df_new_for_merge.merge(
    df_existing_for_merge, on = 'Verse_ID', how = 'left')
df_updated

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Former_Verse,Tests,Best_WPM
0,1,OT,Genesis,1,1,1,Genesis_1:1,"In the beginning, God created heaven and earth.",47,"In the beginning, God created heaven and earth.",29,200.543770
1,2,OT,Genesis,1,1,2,Genesis_1:2,"But the earth was empty and unoccupied, and da...",141,"But the earth was empty and unoccupied, and da...",7,161.088381
2,3,OT,Genesis,1,1,3,Genesis_1:3,"And God said, ""Let there be light."" And light ...",53,"And God said, ""Let there be light."" And light ...",7,157.810945
3,4,OT,Genesis,1,1,4,Genesis_1:4,"And God saw the light, that it was good; and s...",89,"And God saw the light, that it was good; and s...",5,162.812696
4,5,OT,Genesis,1,1,5,Genesis_1:5,"And he called the light, 'Day,' and the darkne...",104,"And he called the light, 'Day,' and the darkne...",2,130.008479
...,...,...,...,...,...,...,...,...,...,...,...,...
35834,35835,NT,Revelation,73,22,17,Revelation_22:17,"And the Spirit and the Bride say: ""Draw near.""...",197,"And the Spirit and the Bride say: ""Draw near.""...",0,0.000000
35835,35836,NT,Revelation,73,22,18,Revelation_22:18,For I call as witnesses all listeners of the w...,176,For I call as witnesses all listeners of the w...,0,0.000000
35836,35837,NT,Revelation,73,22,19,Revelation_22:19,And if anyone will have taken away from the wo...,217,And if anyone will have taken away from the wo...,2,136.493900
35837,35838,NT,Revelation,73,22,20,Revelation_22:20,"He who offers testimony to these things, says:...",108,"He who offers testimony to these things, says:...",2,125.198421


## Creating several new columns to help analyze differences between these two files:

In [7]:
df_updated['Verses_Match'] = (df_updated["Verse"] == df_updated[
"Former_Verse"]).astype('int')
df_updated

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Former_Verse,Tests,Best_WPM,Verses_Match
0,1,OT,Genesis,1,1,1,Genesis_1:1,"In the beginning, God created heaven and earth.",47,"In the beginning, God created heaven and earth.",29,200.543770,1
1,2,OT,Genesis,1,1,2,Genesis_1:2,"But the earth was empty and unoccupied, and da...",141,"But the earth was empty and unoccupied, and da...",7,161.088381,1
2,3,OT,Genesis,1,1,3,Genesis_1:3,"And God said, ""Let there be light."" And light ...",53,"And God said, ""Let there be light."" And light ...",7,157.810945,1
3,4,OT,Genesis,1,1,4,Genesis_1:4,"And God saw the light, that it was good; and s...",89,"And God saw the light, that it was good; and s...",5,162.812696,1
4,5,OT,Genesis,1,1,5,Genesis_1:5,"And he called the light, 'Day,' and the darkne...",104,"And he called the light, 'Day,' and the darkne...",2,130.008479,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35834,35835,NT,Revelation,73,22,17,Revelation_22:17,"And the Spirit and the Bride say: ""Draw near.""...",197,"And the Spirit and the Bride say: ""Draw near.""...",0,0.000000,1
35835,35836,NT,Revelation,73,22,18,Revelation_22:18,For I call as witnesses all listeners of the w...,176,For I call as witnesses all listeners of the w...,0,0.000000,1
35836,35837,NT,Revelation,73,22,19,Revelation_22:19,And if anyone will have taken away from the wo...,217,And if anyone will have taken away from the wo...,2,136.493900,1
35837,35838,NT,Revelation,73,22,20,Revelation_22:20,"He who offers testimony to these things, says:...",108,"He who offers testimony to these things, says:...",2,125.198421,1


Seeing how many verses do *not* match exactly:

In [8]:
differing_verses = len(df_updated.query("Verses_Match == 0").copy())
print(f"{differing_verses} verse(s) \
(with the same Verse ID) does/do not match exactly. This equals \
{round(100*differing_verses/len(df_updated), 3)} percent of all verses.")

88 verse(s) (with the same Verse ID) does/do not match exactly. This equals 0.246 percent of all verses.


Using [TheFuzz](https://github.com/seatgeek/thefuzz?tab=readme-ov-file) to determine the similarity ratio between our current and former verses:

(I found that certain verses can have a similarity ratio of 100 even though they're not the same--possibly due to rounding? Thus, we shouldn't use similarity ratios on their own to determine whether or not two verses are identical.)

In [9]:
df_updated['similarity_ratio'] = [fuzz.ratio(
df_updated.iloc[i]['Verse'], df_updated.iloc[i][
'Former_Verse']) for i in range(len(df_updated))]

# Saving df_updated as a .csv file for reference, as we'll soon remove
# outdated test results and former verses from this file:
df_updated.to_csv('Existing_New_Verse_Comparison.csv', index = False)

Evaluating similarity-ratio percentiles: (Low numbers can indicate that verses underwent major changes and/or something incorrect occurred with the merge.)

In [10]:
print("Similarity ratio quantiles for all verses:")
print(df_updated['similarity_ratio'].quantile([0, 0.1, 0.2, 0.5, 0.8, 0.9, 1]))

Similarity ratio quantiles for all verses:
0.0     92.0
0.1    100.0
0.2    100.0
0.5    100.0
0.8    100.0
0.9    100.0
1.0    100.0
Name: similarity_ratio, dtype: float64


Creating a DataFrame of verses that differ at least slightly from one another:

In [11]:
df_differences = df_updated.query("Verses_Match == 0").copy()
# Saving this as a separate .csv file for further analysis if needed:
df_differences.to_csv('Existing_New_Verse_Differences.csv', index = False)
df_differences

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Former_Verse,Tests,Best_WPM,Verses_Match,similarity_ratio
1905,1906,OT,Exodus,2,14,19,Exodus_14:19,"And the Angel of God, who preceded the camp of...",164,"And the Angel of God, who preceded the camp of...",1,111.754279,0,100
2646,2647,OT,Exodus,2,38,16,Exodus_38:16,All the hangings of the atrium were woven from...,66,All the hangings of the atrium were woven from...,0,0.000000,0,99
4151,4152,OT,Numbers,4,15,5,Numbers_15:5,"and he shall give the same measure of wine, po...",107,"and he shall give the same measure of wine, po...",0,0.000000,0,100
5513,5514,OT,Deuteronomy,5,23,19,Deuteronomy_23:19,"You shall not lend money, or grain, or anythin...",89,"You shall not lend money, or grain, or anythin...",0,0.000000,0,99
5969,5970,OT,Joshua,6,6,24,Joshua_6:24,Then they set fire to the city and all the thi...,189,Then they set fire to the city and all the thi...,0,0.000000,0,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33952,33953,NT,Ephesians,56,1,9,Ephesians_1:9,So does he make known to us the mystery of his...,120,So does he make known to us the mystery of his...,0,0.000000,0,100
34406,34407,NT,2-Thessalonians,60,2,8,2-Thessalonians_2:8,And then that iniquitous one shall be revealed...,176,And then that iniquitous one shall be revealed...,0,0.000000,0,100
35358,35359,NT,1-John,69,4,19,1-John_4:19,"Therefore, let us love God, for God first love...",51,"Therefore, let us love God, for God first love...",0,0.000000,0,99
35476,35477,NT,Revelation,73,2,23,Revelation_2:23,"And I will put her sons to death, and all the ...",199,"And I will put her sons to death, and all the ...",0,0.000000,0,100


Examining similarity-ratio percentiles for differing verses only:

In [12]:
print("Similarity ratio quantiles for differing verses only:")
print(df_differences['similarity_ratio'].quantile([0, 0.1, 0.2, 0.5, 0.8, 0.9, 1]))

Similarity ratio quantiles for differing verses only:
0.0     92.0
0.1     96.0
0.2     96.0
0.5     98.0
0.8     99.6
0.9    100.0
1.0    100.0
Name: similarity_ratio, dtype: float64


Determining how many tests have been typed--and how many verses have been typed at least once: (We'll then compare these results to those in our updated copy in order to see how much test data will be lost in the process of making these updates.)

In [13]:
previous_count_of_verses_typed_at_least_once = len(df_updated.query("Tests > 0"))
previous_verse_count = df_updated['Tests'].sum()


In [14]:
# Removing WPM and test counts for verses that have now been changed:
# (This step could potentially be left out, but I think it will be best 
# for all test metrics to reflect the most recent copies of each verse.)
df_updated['Best_WPM'] = np.where(df_updated['Verses_Match'] == 0, 0.0,
df_updated['Best_WPM'])
df_updated['Tests'] = np.where(df_updated['Verses_Match'] == 0, 0,
df_updated['Tests'])


df_updated

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Former_Verse,Tests,Best_WPM,Verses_Match,similarity_ratio
0,1,OT,Genesis,1,1,1,Genesis_1:1,"In the beginning, God created heaven and earth.",47,"In the beginning, God created heaven and earth.",29,200.543770,1,100
1,2,OT,Genesis,1,1,2,Genesis_1:2,"But the earth was empty and unoccupied, and da...",141,"But the earth was empty and unoccupied, and da...",7,161.088381,1,100
2,3,OT,Genesis,1,1,3,Genesis_1:3,"And God said, ""Let there be light."" And light ...",53,"And God said, ""Let there be light."" And light ...",7,157.810945,1,100
3,4,OT,Genesis,1,1,4,Genesis_1:4,"And God saw the light, that it was good; and s...",89,"And God saw the light, that it was good; and s...",5,162.812696,1,100
4,5,OT,Genesis,1,1,5,Genesis_1:5,"And he called the light, 'Day,' and the darkne...",104,"And he called the light, 'Day,' and the darkne...",2,130.008479,1,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35834,35835,NT,Revelation,73,22,17,Revelation_22:17,"And the Spirit and the Bride say: ""Draw near.""...",197,"And the Spirit and the Bride say: ""Draw near.""...",0,0.000000,1,100
35835,35836,NT,Revelation,73,22,18,Revelation_22:18,For I call as witnesses all listeners of the w...,176,For I call as witnesses all listeners of the w...,0,0.000000,1,100
35836,35837,NT,Revelation,73,22,19,Revelation_22:19,And if anyone will have taken away from the wo...,217,And if anyone will have taken away from the wo...,2,136.493900,1,100
35837,35838,NT,Revelation,73,22,20,Revelation_22:20,"He who offers testimony to these things, says:...",108,"He who offers testimony to these things, says:...",2,125.198421,1,100


Comparing verse completion and test count data in the old and new datasets:

In [15]:
new_count_of_verses_typed_at_least_once = len(
df_updated.query("Tests > 0"))
new_verse_count = df_updated['Tests'].sum()

print(f"{previous_count_of_verses_typed_at_least_once} \
verse(s) had been typed \
at least once in the original dataset, and {previous_verse_count} tests \
was/were present.")

print(f"{new_count_of_verses_typed_at_least_once} \
verse(s) has/have been typed \
at least once in the new dataset, and {new_verse_count} test(s) \
is/are present.")

print(f"In the revised dataset, {
previous_count_of_verses_typed_at_least_once 
- new_count_of_verses_typed_at_least_once} fewer verse(s) have/has been \
typed at least once, and {previous_verse_count - new_verse_count} fewer test(s) is/are present.")




1962 verse(s) had been typed at least once in the original dataset, and 2116 tests was/were present.
1961 verse(s) has/have been typed at least once in the new dataset, and 2115 test(s) is/are present.
In the revised dataset, 1 fewer verse(s) have/has been typed at least once, and 1 fewer test(s) is/are present.


Confirming that these changes had the expected impact: (All Tests and Best_WPM values should now be 0 and 0.0, respectively.)

In [16]:
df_updated.query("Verses_Match == 0")

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Former_Verse,Tests,Best_WPM,Verses_Match,similarity_ratio
1905,1906,OT,Exodus,2,14,19,Exodus_14:19,"And the Angel of God, who preceded the camp of...",164,"And the Angel of God, who preceded the camp of...",0,0.0,0,100
2646,2647,OT,Exodus,2,38,16,Exodus_38:16,All the hangings of the atrium were woven from...,66,All the hangings of the atrium were woven from...,0,0.0,0,99
4151,4152,OT,Numbers,4,15,5,Numbers_15:5,"and he shall give the same measure of wine, po...",107,"and he shall give the same measure of wine, po...",0,0.0,0,100
5513,5514,OT,Deuteronomy,5,23,19,Deuteronomy_23:19,"You shall not lend money, or grain, or anythin...",89,"You shall not lend money, or grain, or anythin...",0,0.0,0,99
5969,5970,OT,Joshua,6,6,24,Joshua_6:24,Then they set fire to the city and all the thi...,189,Then they set fire to the city and all the thi...,0,0.0,0,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33952,33953,NT,Ephesians,56,1,9,Ephesians_1:9,So does he make known to us the mystery of his...,120,So does he make known to us the mystery of his...,0,0.0,0,100
34406,34407,NT,2-Thessalonians,60,2,8,2-Thessalonians_2:8,And then that iniquitous one shall be revealed...,176,And then that iniquitous one shall be revealed...,0,0.0,0,100
35358,35359,NT,1-John,69,4,19,1-John_4:19,"Therefore, let us love God, for God first love...",51,"Therefore, let us love God, for God first love...",0,0.0,0,99
35476,35477,NT,Revelation,73,2,23,Revelation_2:23,"And I will put her sons to death, and all the ...",199,"And I will put her sons to death, and all the ...",0,0.0,0,100


## Revising df_updated so that its columns, and their order, both match that of df_new:

In [17]:
df_updated = df_updated[headers].copy()
df_updated

,Verse_ID,OT_NT,Book,Book_Num,Chapter_Num,Verse_Num,Verse_Code,Verse,Characters,Tests,Best_WPM
0,1,OT,Genesis,1,1,1,Genesis_1:1,"In the beginning, God created heaven and earth.",47,29,200.543770
1,2,OT,Genesis,1,1,2,Genesis_1:2,"But the earth was empty and unoccupied, and da...",141,7,161.088381
2,3,OT,Genesis,1,1,3,Genesis_1:3,"And God said, ""Let there be light."" And light ...",53,7,157.810945
3,4,OT,Genesis,1,1,4,Genesis_1:4,"And God saw the light, that it was good; and s...",89,5,162.812696
4,5,OT,Genesis,1,1,5,Genesis_1:5,"And he called the light, 'Day,' and the darkne...",104,2,130.008479
...,...,...,...,...,...,...,...,...,...,...,...
35834,35835,NT,Revelation,73,22,17,Revelation_22:17,"And the Spirit and the Bride say: ""Draw near.""...",197,0,0.000000
35835,35836,NT,Revelation,73,22,18,Revelation_22:18,For I call as witnesses all listeners of the w...,176,0,0.000000
35836,35837,NT,Revelation,73,22,19,Revelation_22:19,And if anyone will have taken away from the wo...,217,2,136.493900
35837,35838,NT,Revelation,73,22,20,Revelation_22:20,"He who offers testimony to these things, says:...",108,2,125.198421


## Saving df_updated to a .csv file that you can then manually copy and paste into your /Files/ folder (ideally after confirming that everyting looks correct):

In [18]:
df_updated.to_csv('CPDB_for_TTTB.csv', index = False)

In [19]:
end_time = time.time()
print(f"{time.ctime()}: Finished running script in {round(
end_time-start_time, 3)} seconds.")

Tue Dec  2 22:03:10 2025: Finished running script in 2.375 seconds.
